# Predicting the next Cambridge 9702 Paper 4 with RAG
**Cambridge International A Level Physics 9702 · Paper 4 · Google Colab (T4 GPU)**

This notebook turns your folder of past Paper 4 question papers and mark schemes into:

1. **A question table** – one row per marked part, such as Q3(b)(ii), with its text, marks, context and mark-scheme answer.
2. **Syllabus tags** – every part is tagged with the syllabus learning outcome it tests, such as "define gravitational potential".
3. **Analysis** – important topics, the most repeated tasks, trends over time, a definitions bank, and the usual shape of a paper.
4. **A prediction** – a blueprint for the next paper. It is tested on past exams first (a *backtest*), so you can see how much better it is than guessing.
5. **A generated practice paper** – an open AI model from Hugging Face, running inside Colab, writes new questions in Cambridge style. It is given real past questions as examples (this is the RAG part). Python checks the marks, the arithmetic, and that nothing is copied. The paper and its mark scheme are saved as Word files.

**How to run:** choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. On a free T4 the first run can take 1–2 hours, mostly the AI model writing questions. Everything the model writes is saved on Drive, so later runs take a few minutes.

| Word | Meaning here |
|---|---|
| Sitting | One exam session, e.g. MJ 2024 (May/June 2024). It contains several variants (papers 41, 42, 43, 44). |
| Learning outcome (LO) | One line of the syllabus, e.g. "13.4a Define gravitational potential …" |
| Embedding | A list of numbers describing the meaning of a text. Similar meanings give similar numbers. |
| RAG | Retrieval-augmented generation: find relevant past questions first, then give them to the AI model as examples. |

## Step 1 · Install libraries, connect Drive, choose settings
**What it does:** installs the libraries it needs, connects your Google Drive, and sets the options used everywhere else.
**Check:** `DATA` must point at your past-papers folder (subfolders are fine). If the shared folder is not in *My Drive*, add a shortcut to it first.

In [ ]:
!pip install -q pymupdf rank-bm25 python-docx sentence-transformers bitsandbytes accelerate "transformers>=5.3"

import json, math, re, hashlib, time, unicodedata
from pathlib import Path
from itertools import chain
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymupdf as fitz
from tqdm.auto import tqdm
from pydantic import BaseModel
from IPython.display import display, Markdown, Image
from google.colab import drive

drive.mount("/content/drive")

# ----- Settings you can change -----
DATA = Path("/content/drive/MyDrive/physics pastpapers")    # past papers: 9702_s24_qp_41.pdf, 9702_s24_ms_41.pdf, ...
WORK = Path("/content/drive/MyDrive/physics_exam_project")  # results are saved here
SYLLABUS_START = 2022             # first exam year with the current syllabus content
TARGET_EXAM = "October/November 2026"
LLM_MODEL = "Qwen/Qwen3.5-9B"     # Hugging Face model; "Qwen/Qwen3.5-4B" is about twice as fast but writes weaker questions
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
CHECK_TAGS_WITH_LLM = True        # the AI double-checks the uncertain syllabus tags (cached after the first run)
MAX_TRIES = 3                     # attempts per generated question
COPY_LIMIT = 0.95                 # a new part this similar to a past part counts as copied

REPORTS = WORK / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
SESSION_CODE = {"m": "FM", "s": "MJ", "w": "ON"}   # Feb/March, May/June, Oct/Nov
SESSION_ORDER = {"m": 0, "s": 1, "w": 2}

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")

def sitting_label(idx):
    """Turn a sitting number (year*3 + session) back into a label such as 'MJ 2024'."""
    return f"{['FM', 'MJ', 'ON'][idx % 3]} {idx // 3}"

print("Data folder found:", DATA.exists())

## Step 2 · Load the AI model from Hugging Face
**What it does:** downloads [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) from Hugging Face (about 19 GB, a few minutes) and loads it in 4-bit so it fits on a T4.
- `generate()` sends one prompt to the model and returns its reply.
- `ask_llm()` is what the rest of the notebook uses. It adds three things:
  - With a `schema`, it asks for a JSON reply, finds the JSON object that has the right fields, repairs small slips (a `null` in place of an empty field, a LaTeX backslash such as `\Omega`), and checks it with pydantic. The prompt shows the model a short example of the JSON instead of the long formal schema (the model used to copy the schema back, which wasted minutes).
  - An unusable reply raises an error that says **why** (for example `parts.0.marks: Input should be a valid integer`), and the full reply is added to `llm_bad_replies.txt` on Drive so you can read it.
  - Every good answer is cached in `llm_cache.json` on Drive, so re-running never repeats the same work. Unusable replies are not cached.

**Check:** the last line prints a one-sentence answer and how long it took. Warnings that `causal_conv1d` or `flash-linear-attention` is "falling back to its reference PyTorch implementation" are normal on a T4; they only mean the model runs slower.

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU, then run again."

def load_llm(dtype):
    """Load the model with 4-bit weights (about 7 GB of GPU memory for the 9B model)."""
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype)
    return Qwen3_5ForConditionalGeneration.from_pretrained(LLM_MODEL, quantization_config=quant,
                                                           dtype=dtype, device_map={"": 0}).eval()   # whole model on the GPU

def generate(prompt, temperature=0.0, max_new_tokens=1500):
    """Send one prompt to the model and return its reply as text."""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=[text], return_tensors="pt").to(llm.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.9) if temperature else dict(do_sample=False)
    with torch.inference_mode():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=processor.tokenizer.eos_token_id, **sampling)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    if len(new_tokens) >= max_new_tokens:
        print(f"  (the reply was cut off at {max_new_tokens} tokens)")
    reply = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", reply, flags=re.S).strip()

processor = AutoProcessor.from_pretrained(LLM_MODEL)
fast_type = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # a T4 is fastest in float16
llm = load_llm(fast_type)
start = time.time()
reply = generate("In one sentence, what is a photon?", max_new_tokens=60)
if fast_type == torch.float16 and not re.search(r"[A-Za-z]{4}", reply):
    print("float16 gave broken output, so the model is reloaded in bfloat16 (slower but safe).")
    del llm
    torch.cuda.empty_cache()
    llm = load_llm(torch.bfloat16)
    reply = generate("In one sentence, what is a photon?", max_new_tokens=60)
print(reply, f"({time.time() - start:.0f} s)")

from pydantic import ValidationError

LLM_CACHE_FILE = WORK / "llm_cache.json"
llm_cache = json.loads(LLM_CACHE_FILE.read_text()) if LLM_CACHE_FILE.exists() else {}
BAD_REPLIES = WORK / "llm_bad_replies.txt"     # unusable replies are saved here so you can see what went wrong

def drop_nulls(x):
    """Remove null values so pydantic uses the defaults (models often write null for an empty field)."""
    if isinstance(x, dict):
        return {k: drop_nulls(v) for k, v in x.items() if v is not None}
    if isinstance(x, list):
        return [drop_nulls(v) for v in x if v is not None]
    return x

def read_json(reply, schema):
    """The JSON object in a reply that has every field the schema needs.
    It skips words or ``` around the JSON, and smaller objects inside it (for example one part of a question)."""
    text = re.sub(r'(\\\\)|\\(?![/"bfnrtu])', lambda m: m[1] or r"\\", reply)   # LaTeX \Omega -> \\Omega, valid JSON
    needed = {name for name, field in schema.model_fields.items() if field.is_required()}
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", text):
        try:
            obj = decoder.raw_decode(text, m.start())[0]
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and needed <= obj.keys():
            return drop_nulls(obj)
    if "{" not in text:
        raise ValueError("the reply has no JSON object")
    raise ValueError(f"no complete JSON object with the fields {sorted(needed)} - the reply may be cut off")

def ask_llm(prompt, schema=None, temperature=0.0, attempt=0, max_new_tokens=1500):
    """Ask the model. With a pydantic `schema`, the reply must be JSON of that shape and is returned parsed."""
    if schema:                                             # the prompt itself shows the JSON format to use
        prompt += "\n\nReply with ONLY one JSON object, written compactly, with no other text."
    key = hashlib.sha256(f"{LLM_MODEL}|{temperature}|{attempt}|{prompt}".encode()).hexdigest()
    reply = llm_cache.get(key) or generate(prompt, temperature, max_new_tokens)
    result = reply
    if schema:
        try:
            result = schema.model_validate(read_json(reply, schema))
        except (ValueError, ValidationError) as e:         # unusable reply: explain why, save it, do NOT cache it
            reason = ("; ".join(f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors()[:3])
                      if isinstance(e, ValidationError) else str(e))
            with open(BAD_REPLIES, "a", encoding="utf-8") as f:
                f.write(f"\n\n===== {schema.__name__}: {reason}\n{reply}")
            raise ValueError(f"{reason}. The reply starts: {reply[:150]!r}") from None
    if key not in llm_cache:
        llm_cache[key] = reply
        save_json(LLM_CACHE_FILE, llm_cache)
    return result

## Step 3 · Find the papers and keep complete pairs
**What it does:** reads year, session and variant from each filename. For example, `9702_s24_qp_41.pdf` is May/June 2024, question paper, variant 41. A paper is kept only when both its question paper (`qp`) and mark scheme (`ms`) exist.

**Check:** the table shows how many papers you have per year and session.

In [ ]:
NAME = re.compile(r"9702_([msw])(\d{2})_(qp|ms)_(4\d)\.pdf", re.I)
files = []
for pdf in sorted(DATA.rglob("*.pdf")):
    m = NAME.fullmatch(pdf.name)
    if m:
        session, yy, kind, comp = m.groups()
        files.append({"paper_id": f"9702_{session.lower()}{yy}_{comp}", "year": 2000 + int(yy),
                      "session": session.lower(), "component": comp, "kind": kind.lower(), "path": str(pdf)})
files = pd.DataFrame(files)
assert not files.empty, f"No 9702 Paper 4 PDFs found in {DATA}"

papers = (files.drop_duplicates(["paper_id", "kind"])
               .pivot(index=["paper_id", "year", "session", "component"], columns="kind", values="path")
               .dropna().reset_index())
print(f"{len(papers)} complete papers; {files.paper_id.nunique() - len(papers)} papers skipped (qp or ms missing)")
display(papers.groupby(["year", "session"]).size().unstack(fill_value=0).rename(columns=SESSION_CODE))

## Step 4 · Functions that read a PDF into question records
**What it does:** defines the PDF reader; it does not run it yet.
- Question papers are read by layout. The main question number sits in a narrow left column, the labels (a) and (i) sit further right, and marks look like `[2]`.
- Mark schemes are read in the PDF's own order, which keeps the answer table rows together.
- Each scored part is matched with its mark-scheme entry.

Improvements over the previous version:
- Powers keep their meaning: `10^5`, `m s^–1`. Subscripts show as `c_0`.
- PapaCambridge watermarks and barcode lines are removed.
- Each part gets its Cambridge command word (Define, Calculate, Show that, …).

In [ ]:
JUNK = re.compile(r"©|UCLES|PapaCambridge|Trace ID|Licensed for|Re-uploading|Downloaded from|DO NOT WRITE|"
                  r"\[Turn over|BLANK PAGE|Section [AB]|^\*\s*\d{6,}|\bDFD\b|9702/\d\d/", re.I)

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"[ \t]+", " ", re.sub(r"\.{4,}|…{2,}", " ", text)).strip()

def line_text(line):
    """Join the pieces (spans) of one PDF line, marking powers with ^ and subscripts with _ (10^5, c_0)."""
    spans = [s for s in line["spans"] if s["text"].strip()]
    if not spans:
        return ""
    main = max(spans, key=lambda s: s["size"] * len(s["text"]))
    out = ""
    for s in line["spans"]:
        t, small = s["text"], s["size"] < 0.85 * main["size"]
        if t.strip() and (s["flags"] & 1 or (small and s["origin"][1] < main["origin"][1] - 1)):
            t = "^" + t.lstrip()
        elif t.strip() and small and s["origin"][1] > main["origin"][1] + 1:
            t = "_" + t.lstrip()
        out += t
    return out

def page_lines(page, pno):
    """Question-paper lines in reading order, without headers, footers, margins and watermarks."""
    W, H = page.rect.width, page.rect.height
    lines = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            x, y = line["bbox"][:2]
            text = clean_text(line_text(line))
            if (not text or not 0.05 * H < y < 0.94 * H or x < 0.04 * W or abs(line["dir"][1]) > 0.1
                    or text in {"For", "Examiner’s", "Examiner's", "Use"} or JUNK.search(text)
                    or sum(ord(c) < 32 for c in text) > 3
                    or sum("Ā" <= c <= "ſ" for c in text) > 2):    # barcode gibberish
                continue
            lines.append({"text": text, "x": x / W, "y": y, "page": pno})
    return sorted(lines, key=lambda l: (round(l["y"] / 3), l["x"]))

def extract_questions(path):
    """One record per scored part, keeping the parent question's shared context."""
    groups, current = {}, None
    with fitz.open(path) as doc:
        for pno in range(2, len(doc) + 1):                       # page 1 is the cover
            for line in page_lines(doc[pno - 1], pno):
                m = re.fullmatch(r"(\d{1,2})\s*(.*)", line["text"])
                if m and 0.075 <= line["x"] <= 0.095 and 1 <= int(m[1]) <= 25:   # main question number
                    number = int(m[1])
                    if current is None and number != 1:
                        continue
                    if current is None or number == current + 1:
                        current = number
                        groups[current] = [{**line, "text": m[2], "x": 0.12}] if m[2] else []
                        continue
                if current is not None:
                    groups[current].append(line)
    records = []
    for number, lines in groups.items():
        nodes, path_ = {(): []}, ()                              # () holds the parent question's context
        for line in lines:
            label = re.match(r"^\(([a-z]+)\)\s*(.*)", line["text"])
            if label and line["x"] < 0.21:
                token, rest = label.groups()
                if len(token) == 1 and line["x"] < 0.145:
                    path_ = (token,)
                    nested = re.match(r"^\(([ivx]+)\)\s*(.*)", rest)
                    if nested:
                        path_ += (nested[1],)
                elif re.fullmatch(r"[ivx]+", token) and path_:
                    path_ = (path_[0], token)
            nodes.setdefault(path_, []).append(line)
        full_text = "\n".join(l["text"] for l in lines)
        for path_, body in nodes.items():
            text = "\n".join(l["text"] for l in body)
            marks = re.findall(r"\[\s*(\d{1,2})\s*\]", text)
            if not marks:
                continue
            context = [l for depth in range(len(path_)) for l in nodes.get(path_[:depth], [])]
            records.append({"part": str(number) + "".join(f"({p})" for p in path_), "question_number": number,
                            "text": text, "marks": sum(map(int, marks)),
                            "context": "\n".join(l["text"] for l in context), "full_question": full_text,
                            "context_pages": sorted({l["page"] for l in context + body}),
                            "damaged": bool(re.search(r"[\x00-\x08\x0b-\x1f-]", full_text))})
    return records, sorted(groups)

MS_JUNK = re.compile(r"©|UCLES|Cambridge|PUBLISHED|Syllabus|Teachers|Page \d|9702|May/June|March 20|October/November|"
                     r"PapaCambridge|Trace ID|Licensed for|Re-uploading|Downloaded from|Section [AB]|\bDFD\b|^\*\s*\d{6,}", re.I)

def extract_mark_scheme(path):
    """Mark-scheme text grouped by part label, e.g. {'1(b)(i)': {...}}."""
    entries, current, question, letter = {}, None, None, None
    with fitz.open(path) as doc:
        for pno in range(2, len(doc) + 1):
            page = doc[pno - 1]
            blocks = page.get_text("dict")["blocks"]
            main_labels = {}                                     # main question numbers in the left column
            for block in blocks:
                for line in block.get("lines", []):
                    token = clean_text(line_text(line))
                    box = fitz.Rect(line["bbox"]) * page.rotation_matrix
                    start = re.match(r"^(\d{1,2})(?:\s+(.*)|$)", token)
                    if start and box.x0 < 0.12 * page.rect.width and 1 <= int(start[1]) <= 25:
                        main_labels[token] = (start[1], start[2] or "")
            for block in blocks:                                 # the PDF's own order keeps table rows together
                for line in block.get("lines", []):
                    text = re.sub(r"^(\d{1,2})([a-z])(\([ivx]+\))(?=\s|$)", r"\1(\2)\3", clean_text(line_text(line)))
                    if not text or MS_JUNK.search(text) or text in {"Question", "Answer", "Marks", "Paper", "Mark Scheme"}:
                        continue
                    full = re.match(r"^(\d{1,2})\s*(\([a-z]\)(?:\s*\([ivx]+\))?)\s*(.*)", text)
                    if full:
                        question, letter = full[1], re.findall(r"\(([a-z])\)", full[2])[0]
                        current, text = question + re.sub(r"\s+", "", full[2]), full[3]
                    elif text in main_labels and int(main_labels[text][0]) > int(question or 0):
                        question, text = main_labels[text]
                        letter, current = None, question
                    elif question and re.match(r"^\([a-z]+\)", text):
                        token, text = re.match(r"^\(([a-z]+)\)\s*(.*)", text).groups()
                        if len(token) == 1 and (token not in {"i", "v", "x"} or letter is None):
                            letter, current = token, f"{question}({token})"
                            nested = re.match(r"^\(([ivx]+)\)\s*(.*)", text)
                            if nested:
                                current, text = current + f"({nested[1]})", nested[2]
                        elif letter and re.fullmatch(r"[ivx]+", token):
                            current = f"{question}({letter})({token})"
                    if current:
                        entry = entries.setdefault(current, {"lines": [], "pages": set()})
                        if text:
                            entry["lines"].append(text)
                        entry["pages"].add(pno)
    return {k: {"text": "\n".join(v["lines"]), "pages": sorted(v["pages"])} for k, v in entries.items()}

def extract_pair(qp, ms):
    """Read one paper, attach mark-scheme answers, and run simple structure checks."""
    records, numbers = extract_questions(qp)
    answers = extract_mark_scheme(ms)
    for r in records:
        part = r["part"]
        answer, match = answers.get(part), "exact"
        if not answer:                                           # e.g. the scheme only has 2(b)(i) and 2(b)(ii)
            kids = [v for k, v in answers.items() if k.startswith(part + "(")]
            answer = {"text": "\n".join(v["text"] for v in kids), "pages": sorted({p for v in kids for p in v["pages"]})} if kids else None
            match = "children" if kids else "missing"
        while answer is None and "(" in part:                    # fall back to the parent entry
            part = part[:part.rfind("(")]
            if part in answers:
                answer, match = answers[part], "parent"
        r.update(answer=answer["text"] if answer else "", answer_pages=answer["pages"] if answer else [], answer_match=match)
    total = sum(r["marks"] for r in records)
    ok = bool(numbers) and numbers == list(range(1, max(numbers) + 1)) and total == 100 \
         and len({r["part"] for r in records}) == len(records)
    return records, {"extraction_ok": ok, "marks": total, "questions": len(numbers), "parts": len(records),
                     "missing_answers": sum(r["answer_match"] == "missing" for r in records)}

COMMANDS = {"show that": "Show that", "state what is meant by": "Define", "define": "Define", "state": "State",
            "explain": "Explain", "describe": "Describe", "suggest": "Suggest", "calculate": "Calculate",
            "determine": "Determine", "derive": "Derive", "deduce": "Deduce", "estimate": "Estimate",
            "sketch": "Sketch", "draw": "Draw", "complete": "Complete", "compare": "Compare", "outline": "Outline",
            "identify": "Identify", "predict": "Predict", "discuss": "Discuss", "plot": "Plot"}
COMMAND_RE = re.compile(r"\b(" + "|".join(k.replace(" ", r"\s+") for k in sorted(COMMANDS, key=len, reverse=True)) + r")\b", re.I)

def command_word(text, context=""):
    """The Cambridge command word of a part. For list parts such as '(i) the radius', it is taken from the stem ('Calculate:')."""
    body = re.sub(r"^\s*(\(\w+\)\s*)+", "", str(text))[:200]
    m = COMMAND_RE.search(body) or COMMAND_RE.search(str(context)[-150:])
    return COMMANDS[" ".join(m[1].lower().split())] if m else "Other"

## Step 5 · Read every paper
**What it does:** runs the reader on all papers (about a minute) and builds `df`, the question table used by everything else. It also adds:
- the **sitting** (e.g. `MJ 2024`), so different variants of the same exam are not counted as repeats;
- the **command word** of each part.

**Check:** a normal Paper 4 totals 100 marks. Papers that fail this check are listed; they are left out of the statistics but can still be searched.

In [ ]:
rows, checks = [], []
for p in tqdm(papers.itertuples(), total=len(papers), desc="Reading papers"):
    try:
        records, check = extract_pair(p.qp, p.ms)
    except Exception as e:                                       # one broken PDF should not stop the run
        records, check = [], {"extraction_ok": False, "error": str(e)}
    rows += [{**r, "paper_id": p.paper_id, "year": p.year, "session": p.session, "component": p.component,
              "qp": p.qp, "ms": p.ms, "extraction_ok": check["extraction_ok"]} for r in records]
    checks.append({"paper_id": p.paper_id, **check})

df = pd.DataFrame(rows)
quality = pd.DataFrame(checks)
df["question_id"] = df.paper_id + "_Q" + df.part
df["sitting_idx"] = df.year * 3 + df.session.map(SESSION_ORDER)
df["sitting"] = df.sitting_idx.map(sitting_label)
df["current"] = df.year >= SYLLABUS_START
df["command"] = [command_word(t, c) for t, c in zip(df.text, df.context)]
quality.to_csv(REPORTS / "extraction_quality.csv", index=False)

print(f"{len(df):,} scored parts from {df.paper_id.nunique()} papers; "
      f"{quality.extraction_ok.sum()}/{len(quality)} papers pass the 100-mark check")
display(quality[~quality.extraction_ok].head(15))

## Step 6 · Look at any question and its original pages
**What it does:** `show_question(id)` prints the extracted text and mark-scheme answer, then shows the real PDF pages. Use it whenever a result looks strange.

**Example:** `show_question("9702_s24_41_Q1(a)")`. Copy any `question_id` from a table below.

In [ ]:
def show_question(question_id, images=True):
    r = df[df.question_id == question_id].iloc[0]
    display(Markdown(f"**{question_id}** · {r.sitting} · {r.marks} marks · command: {r.command} · tag: {r.get('lo', '-')}"))
    print("CONTEXT\n", r.context, "\n\nQUESTION\n", r.text, f"\n\nMARK SCHEME ({r.answer_match})\n", r.answer or "(not found)")
    if images:
        for path, pages in [(r.qp, r.context_pages), (r.ms, r.answer_pages)]:
            with fitz.open(path) as doc:
                for p in pages:
                    display(doc[p - 1].get_pixmap(dpi=80).pil_image())

show_question(df.question_id.iat[0], images=False)

## Step 7 · The syllabus learning outcomes
**What it does:** loads the A Level learning outcomes (topics 12–25). These are what each question part gets tagged with.
- The list is paraphrased from the 9702 syllabus for 2025–2027. The letter at the end of each code (13.4**a**) is just its order inside the sub-topic.
- Codes starting with **L** are old topics that only appear in papers before 2022.
- On the first run the list is saved to `syllabus_outcomes.csv` on Drive. You can edit that file, for example to match the official wording; delete it to get the defaults back.

In [ ]:
DEFAULT_OUTCOMES = """code|topic|outcome
12.1a|Motion in a circle|Define the radian and express angles and angular displacement in radians
12.1b|Motion in a circle|Use angular speed: omega = 2 pi / T = 2 pi f and the relation v = r omega for motion in a circle
12.2a|Motion in a circle|Explain that an object moving in a circle at constant speed has an acceleration and a resultant force directed towards the centre
12.2b|Motion in a circle|Calculate centripetal acceleration using a = r omega^2 or a = v^2 / r
12.2c|Motion in a circle|Calculate centripetal force F = m r omega^2 = m v^2 / r and identify which force (tension, friction, weight, normal contact force, lift) provides it, e.g. bends, loops, conical pendulum
13.1a|Gravitational fields|Define gravitational field strength as force per unit mass and represent a gravitational field with field lines
13.2a|Gravitational fields|Treat a uniform sphere as a point mass at its centre and use Newton's law of gravitation F = G m1 m2 / r^2
13.2b|Gravitational fields|Analyse circular orbits where gravitational force provides the centripetal force: orbital speed, period, radius, show that T^2 is proportional to r^3, mass of a planet
13.2c|Gravitational fields|Describe geostationary orbits (equatorial, period 24 hours, same direction as the Earth's rotation) and calculate their radius or height
13.3a|Gravitational fields|Derive and use g = G M / r^2 for the field of a point mass; g approximately constant near the Earth's surface; graphs of g against distance
13.4a|Gravitational fields|Define gravitational potential at a point as the work done per unit mass in bringing a small test mass from infinity to the point; explain why it is negative
13.4b|Gravitational fields|Use gravitational potential phi = - G M / r; field strength equals minus the potential gradient; potential due to two bodies
13.4c|Gravitational fields|Use gravitational potential energy Ep = - G M m / r; changes in potential energy, energy needed to move a satellite or mass, kinetic and total energy in orbit, escape
14.1a|Temperature|Thermal energy is transferred from higher to lower temperature; regions at the same temperature are in thermal equilibrium
14.2a|Temperature|A physical property that varies with temperature can be used to measure temperature; compare thermometers such as thermocouple, liquid-in-glass, resistance and gas thermometers
14.2b|Temperature|Thermodynamic (kelvin) scale: absolute zero, independent of the property of any substance; convert T / K = theta / degrees C + 273.15
14.3a|Temperature|Define and use specific heat capacity c in E = m c delta theta, including experiments and temperature-time graphs
14.3b|Temperature|Define and use specific latent heat of fusion and of vaporisation E = m L; explain the difference in terms of molecular separation and energy; experiments
15.1a|Ideal gases|Amount of substance in moles and the Avogadro constant NA; number of molecules, molar mass, number of moles
15.2a|Ideal gases|Equation of state for an ideal gas p V = n R T: calculate pressure, volume, temperature or amount of gas and changes of state
15.2b|Ideal gases|Use p V = N k T with the Boltzmann constant k = R / NA; calculate the number of molecules N in a gas
15.3a|Ideal gases|State the basic assumptions of the kinetic theory of gases (random motion, elastic collisions, negligible molecular volume, no intermolecular forces); state what is meant by an ideal gas
15.3b|Ideal gases|Explain how molecular collisions with the walls cause gas pressure (change in momentum, force); derive p V = 1/3 N m <c^2>
15.3c|Ideal gases|Calculate the root-mean-square (r.m.s.) speed and mean square speed of gas molecules
15.3d|Ideal gases|Show that the mean translational kinetic energy of a molecule is 3/2 k T and is proportional to thermodynamic temperature; r.m.s. speed at a given temperature
16.1a|Thermodynamics|Define internal energy as the sum of the random distribution of kinetic and potential energies of the molecules; internal energy of an ideal gas; link to temperature
16.2a|Thermodynamics|Work done by or on a gas at constant pressure W = p delta V; work from the area under a p-V graph
16.2b|Thermodynamics|First law of thermodynamics delta U = q + W with correct signs; apply to cycles and changes at constant volume, constant pressure, isothermal and adiabatic; complete tables of energy changes
17.1a|Oscillations|Use displacement, amplitude, period, frequency, angular frequency omega = 2 pi f and phase difference for oscillations
17.1b|Oscillations|Define simple harmonic motion: acceleration proportional to displacement and directed towards a fixed point, a = - omega^2 x; show a system (spring, pendulum, liquid in tube) moves with s.h.m.
17.1c|Oscillations|Use x = x0 sin omega t, v = v0 cos omega t and v = omega root(x0^2 - x^2); graphs of displacement, velocity and acceleration; maximum speed and acceleration
17.2a|Oscillations|Interchange between kinetic and potential energy in s.h.m.; total energy E = 1/2 m omega^2 x0^2; energy against displacement graphs
17.3a|Oscillations|Damping (light, critical, heavy) and its effect on amplitude; graphs of damped oscillations; examples
17.3b|Oscillations|Forced oscillations and resonance: maximum amplitude when driving frequency equals natural frequency; effect of damping on the resonance curve; useful and unwanted resonance
18.1a|Electric fields|Define electric field strength as force per unit positive charge; electric field lines and field patterns between charges and plates
18.2a|Electric fields|Use E = delta V / delta d for the uniform field between charged parallel plates
18.2b|Electric fields|Motion of charged particles in a uniform electric field: force, acceleration, parabolic path, deflection, balanced forces on a charged drop
18.3a|Electric fields|A uniformly charged sphere acts as a point charge at its centre; Coulomb's law F = Q1 Q2 / (4 pi eps0 r^2)
18.4a|Electric fields|Field strength of a point charge E = Q / (4 pi eps0 r^2); combined field of two charges, neutral point; graphs of E against distance
18.5a|Electric fields|Define electric potential at a point as the work done per unit positive charge in bringing a small test charge from infinity to the point
18.5b|Electric fields|Field strength equals minus the potential gradient; V = Q / (4 pi eps0 r); potential of two or more charges as a scalar sum; graphs of V against distance
18.5c|Electric fields|Electric potential energy Ep = Q q / (4 pi eps0 r); changes in potential and kinetic energy of charged particles, closest approach of an alpha particle
19.1a|Capacitance|Define capacitance C = Q / V for an isolated spherical conductor and for a parallel plate capacitor; the farad; isolated sphere C = 4 pi eps0 r
19.1b|Capacitance|Derive and use formulae for capacitors in series and in parallel; charge and potential difference in capacitor networks
19.2a|Capacitance|Energy stored in a capacitor from the area under a potential difference-charge graph; W = 1/2 Q V = 1/2 C V^2
19.3a|Capacitance|Graphs of potential difference, charge and current against time for a capacitor discharging through a resistor; x = x0 exp(- t / R C)
19.3b|Capacitance|Time constant tau = R C: calculate it, find it from a graph, use it to find time, charge or p.d. during discharge
20.1a|Magnetic fields|A magnetic field is a field of force produced by moving charges or permanent magnets; magnetic field lines
20.2a|Magnetic fields|Force on a current-carrying conductor F = B I L sin theta; Fleming's left-hand rule; current balance
20.2b|Magnetic fields|Define magnetic flux density as force per unit current per unit length on a wire at right angles to the field; the tesla
20.3a|Magnetic fields|Force on a moving charge F = B Q v sin theta; circular path of charged particles in a magnetic field r = m v / B Q; mass spectrometer, particle tracks
20.3b|Magnetic fields|Hall voltage: explain its origin, derive and use V_H = B I / (n t q); Hall probe; effect of charge carrier density
20.3c|Magnetic fields|Crossed electric and magnetic fields as a velocity selector v = E / B; deflection of charged particles; specific charge e / m
20.4a|Magnetic fields|Field patterns of a long straight wire, a flat circular coil and a long solenoid; effect of an iron core; forces between parallel current-carrying wires
20.5a|Magnetic fields|Define magnetic flux as the product of flux density and the area perpendicular to the field; the weber; magnetic flux linkage N phi
20.5b|Magnetic fields|Faraday's law: induced e.m.f. equals the rate of change of magnetic flux linkage; calculate induced e.m.f. for moving or rotating conductors and changing fields
20.5c|Magnetic fields|Lenz's law and the direction of induced e.m.f. or current; experiments with magnets and coils; conservation of energy
21.1a|Alternating currents|Period, frequency, peak value and root-mean-square (r.m.s.) value of an alternating current or voltage; x = x0 sin omega t; graphs against time
21.1b|Alternating currents|Mean power in a resistive load is half the maximum power for sinusoidal a.c.; I_rms = I0 / root 2 and V_rms = V0 / root 2; r.m.s. of other waveforms
21.2a|Alternating currents|Half-wave rectification with one diode and full-wave rectification with a bridge of four diodes; draw rectifier circuits
21.2b|Alternating currents|Smoothing with a capacitor; effect of capacitance and load resistance on ripple; sketch smoothed output
22.1a|Quantum physics|A photon is a quantum of electromagnetic energy; E = h f = h c / lambda; the electronvolt
22.1b|Quantum physics|Momentum of a photon p = E / c; force or pressure from absorbed or reflected photons
22.2a|Quantum physics|The photoelectric effect: photoelectrons, threshold frequency and threshold wavelength; evidence for the particulate nature of light
22.2b|Quantum physics|Photoelectric equation h f = work function + 1/2 m v_max^2; define work function; graphs of maximum kinetic energy against frequency; effect of intensity
22.3a|Quantum physics|Electron diffraction as evidence for the wave nature of particles; wave-particle duality
22.3b|Quantum physics|De Broglie wavelength lambda = h / p; wavelength of electrons accelerated through a potential difference
22.4a|Quantum physics|Discrete electron energy levels in isolated atoms; emission and absorption line spectra; energy level diagrams
22.4b|Quantum physics|h f = E1 - E2 for photon emission or absorption between energy levels; wavelengths of spectral lines; number of possible transitions
23.1a|Nuclear physics|Mass-energy equivalence E = m c^2; define mass defect and binding energy; calculate them using unified atomic mass units
23.1b|Nuclear physics|Binding energy per nucleon against nucleon number; relevance to fusion, fission and nuclear stability
23.1c|Nuclear physics|Energy released in nuclear reactions (fission, fusion, alpha or beta decay) from mass changes; nuclear equations
23.2a|Nuclear physics|Radioactive decay is spontaneous and random; fluctuations in count rate as evidence of randomness
23.2b|Nuclear physics|Define activity and decay constant; A = lambda N; define half-life; lambda = 0.693 / half-life
23.2c|Nuclear physics|Exponential decay x = x0 exp(- lambda t) for activity, number of nuclei or count rate; decay graphs and logarithmic graphs
24.1a|Medical physics|Production and detection of ultrasound with a piezoelectric transducer
24.1b|Medical physics|Use of ultrasound to obtain information about internal structures: reflection at boundaries, time delay to find depth
24.1c|Medical physics|Specific acoustic impedance Z = rho c; intensity reflection coefficient (Z1 - Z2)^2 / (Z1 + Z2)^2; use of a coupling gel
24.1d|Medical physics|Attenuation of ultrasound or X-rays in matter I = I0 exp(- mu x); linear attenuation coefficient; half-value thickness
24.2a|Medical physics|Production of X-rays by bombarding a metal target with high-speed electrons; minimum wavelength from the accelerating potential difference; X-ray spectrum
24.2b|Medical physics|X-ray imaging of internal structures: sharpness and contrast of the image; different attenuation by bone and soft tissue
24.2c|Medical physics|Computed tomography (CT): a 3-D image built from many 2-D X-ray images taken at different angles; voxels
24.3a|Medical physics|PET scanning: a tracer containing radioactive nuclei is introduced into the body and absorbed by the tissue being studied
24.3b|Medical physics|Positrons from beta-plus decay annihilate with electrons producing two gamma photons in opposite directions; energy of annihilation photons from mass-energy
24.3c|Medical physics|Detection of gamma photons and time delays to produce a PET image of tracer concentration
25.1a|Astronomy and cosmology|Luminosity of a star; radiant flux intensity F = L / (4 pi d^2) and the inverse square law
25.1b|Astronomy and cosmology|Standard candles (objects of known luminosity such as Cepheid variables and type 1a supernovae) to find distances to galaxies
25.2a|Astronomy and cosmology|Wien's displacement law: wavelength of maximum emission is inversely proportional to thermodynamic temperature; surface temperature of stars
25.2b|Astronomy and cosmology|Stefan-Boltzmann law L = 4 pi sigma r^2 T^4; use with Wien's law to estimate the radius of a star; compare stars
25.3a|Astronomy and cosmology|Redshift of spectral lines from distant galaxies; Doppler redshift delta lambda / lambda = delta f / f = v / c; speed of recession
25.3b|Astronomy and cosmology|Hubble's law v = H0 d; redshift as evidence for an expanding Universe and the Big Bang theory; age of the Universe
L1a|Electronics (legacy)|Ideal operational amplifier properties, open-loop gain and the comparator
L1b|Electronics (legacy)|Inverting and non-inverting amplifiers, negative feedback, voltage gain formulae
L1c|Electronics (legacy)|Sensors (light-dependent resistor, thermistor, strain gauge, piezo-electric transducer) in potential divider circuits; output devices relay, LED, meter
L2a|Communication (legacy)|Amplitude and frequency modulation, carrier wave, sidebands and bandwidth
L2b|Communication (legacy)|Analogue and digital signals, sampling, analogue-to-digital conversion, bits, regeneration
L2c|Communication (legacy)|Communication channels: wire pairs, coaxial cable, optic fibre, microwave links, satellites, mobile phones; attenuation in decibels
L3a|Other legacy topics|Magnetic resonance imaging (MRI): Larmor frequency, radio-frequency pulses, relaxation, non-uniform field
L3b|Other legacy topics|Older syllabus content not in the current A Level list, e.g. measurement of specific charge or Millikan's experiment
"""
SYLLABUS_FILE = WORK / "syllabus_outcomes.csv"
if not SYLLABUS_FILE.exists():
    SYLLABUS_FILE.write_text(DEFAULT_OUTCOMES.strip())
los = pd.read_csv(SYLLABUS_FILE, sep="|", dtype=str)
TOPIC_ORDER = los.topic.drop_duplicates().tolist()
LEGACY_TOPICS = set(los.topic[los.code.str.startswith("L")])
print(f"{len(los)} learning outcomes in {len(TOPIC_ORDER)} topics")

## Step 8 · Embeddings, syllabus tags and search
**What it does:**
1. Turns every part (context + question + mark scheme) into an **embedding**. These are cached on Drive.
2. Tags each part with its closest **learning outcome**. Papers from 2022 onward can only get current outcomes.
3. *(Optional)* The AI model checks the **uncertain** tags, where the two best outcomes score almost the same. It sees 10 parts at a time, each with its top 3 outcomes, and replies with one letter (A, B or C) per part. If a reply does not have the right number of letters, that batch keeps its embedding tags. This only runs for papers from 2022 onward.
4. Defines `search()` (meaning + exact words, i.e. hybrid RAG search) and `similar()`.

**Check:** look at the random sample at the end. Do the tags make sense? Note how many you would change; that is the tag accuracy you can report.

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

embedder = SentenceTransformer(EMBED_MODEL, device="cuda").half()      # small model, shares the GPU with the LLM
embedder.max_seq_length = 512

def embed(texts, prompt=None):
    """Meaning vectors (length 1) for a list of texts."""
    return np.asarray(embedder.encode(list(texts), prompt=prompt, batch_size=32, normalize_embeddings=True), dtype=np.float32)

def squash(text, n=None):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:n] if n else text

def part_text(context, text, answer):
    return f"Context: {squash(context)[-400:]}\nQuestion: {squash(text)}\nMark scheme: {squash(answer, 400)}"

df["search_text"] = [part_text(c, t, a) for c, t, a in zip(df.context, df.text, df.answer)]

# 1. Embeddings of all parts (recomputed only when the extracted text changes)
VEC_FILE = WORK / "part_vectors.npz"
text_hash = hashlib.sha256("\n".join(df.search_text).encode()).hexdigest()
part_vecs = None
if VEC_FILE.exists():
    with np.load(VEC_FILE) as saved:
        if str(saved["hash"]) == text_hash:
            part_vecs = saved["vecs"]
if part_vecs is None:
    part_vecs = embed(df.search_text)
    np.savez(VEC_FILE, vecs=part_vecs, hash=text_hash)

# 2. Tag each part with the closest learning outcome
LO_PROMPT = "Instruct: Find physics exam questions that assess this syllabus learning outcome\nQuery: "
lo_vecs = embed(los.outcome, prompt=LO_PROMPT)
sims = part_vecs @ lo_vecs.T
sims[np.ix_(df.current.values, los.code.str.startswith("L").values)] = -1    # no legacy tags for current papers
top3 = np.argsort(-sims, axis=1)[:, :3]
tags = los.code.values[top3[:, 0]].copy()

# 3. Optional: when the top two outcomes are close, the AI model picks the best of the top 3 (10 parts per request)
rows = np.arange(len(df))
unsure = sims[rows, top3[:, 0]] - sims[rows, top3[:, 1]] < 0.04
if CHECK_TAGS_WITH_LLM:
    todo = np.flatnonzero(df.current.values & unsure)
    print(f"{len(todo)} of {int(df.current.sum())} current parts have an uncertain tag; the AI checks those.")
    changed, kept = 0, 0
    for start in tqdm(range(0, len(todo), 10), desc="AI checks syllabus tags"):
        batch = todo[start:start + 10]
        prompt = ("For each Cambridge A Level Physics exam question part below, choose the syllabus learning outcome "
                  f"(A, B or C) that it mainly assesses.\nReply with ONLY {len(batch)} letters separated by spaces, "
                  "one per part in order, for example: A C B\n")
        for n, i in enumerate(batch, 1):
            options = "\n".join(f"{'ABC'[j]}) {los.outcome.iat[top3[i, j]]}" for j in range(3))
            prompt += f"\nPart {n}:\n{df.search_text.iat[i]}\n{options}\n"
        reply = ask_llm(prompt, max_new_tokens=60)
        picks = re.findall(r"\b[ABC]\b", reply)              # works for "A C B", "1. A 2. C" or ["A", "C"]
        if len(picks) == len(batch):
            new = [los.code.iat[top3[i, "ABC".index(c)]] for i, c in zip(batch, picks)]
            changed += int((tags[batch] != new).sum())
            tags[batch] = new
        else:
            kept += len(batch)
            print(f"A batch kept its embedding tags: expected {len(batch)} letters, the reply was {reply[:80]!r}")
    print(f"The AI changed {changed} tags; {kept} parts kept their embedding tag.")

info = los.set_index("code")
df["lo"], df["topic"], df["lo_text"] = tags, info.topic.reindex(tags).values, info.outcome.reindex(tags).values
df.drop(columns=["search_text"]).to_parquet(WORK / "questions_tagged.parquet")

# 4. Search tools
bm25 = BM25Okapi([re.findall(r"\w+", t.lower()) for t in df.search_text])
QUERY_PROMPT = "Instruct: Given a question about physics exams, retrieve relevant exam question parts\nQuery: "

def search(query, years=(SYLLABUS_START, 2100), k=6):
    """Hybrid search: meaning (embeddings) + exact words (BM25), combined by rank."""
    ok = df.year.between(*years).values
    q = embed([query], prompt=QUERY_PROMPT)[0]
    score = np.zeros(len(df))
    for s in (part_vecs @ q, bm25.get_scores(re.findall(r"\w+", query.lower()))):
        top = np.argsort(-np.where(ok, s, -np.inf))[:100]
        score[top] += 1 / (60 + np.arange(1, len(top) + 1))
    best = [i for i in np.argsort(-score) if ok[i]][:k]
    return df.iloc[best]

def similar(question_id, k=8):
    """The most similar parts from other sittings."""
    i = int(np.flatnonzero(df.question_id.values == question_id)[0])
    s = part_vecs @ part_vecs[i]
    best = [j for j in np.argsort(-s) if df.sitting_idx.iat[j] != df.sitting_idx.iat[i]][:k]
    return df.iloc[best].assign(similarity=s[best].round(3))[["question_id", "sitting", "lo", "command", "marks", "similarity", "text"]]

print("Parts per topic (current syllabus):")
display(df[df.current].topic.value_counts().to_frame("parts"))
with pd.option_context("display.max_colwidth", 90):
    display(df[df.current].sample(min(10, int(df.current.sum())), random_state=1)[["question_id", "text", "lo", "lo_text"]])

## Step 9 · Analysis: important topics, repeated tasks and trends
**What it does:** everything below is plain counting with pandas, so it is fast, complete and repeatable. It uses papers from `SYLLABUS_START` onward that pass the 100-mark check.

| Output | Question it answers |
|---|---|
| Marks per topic (bar chart and heatmap by year) | Which topics carry the most marks, and is that changing? |
| Most tested learning outcomes | Which specific tasks come back, how often, and when were they last seen? |
| Repeated tasks (outcome + command word) | e.g. "Define gravitational potential" in 9 of 14 sittings |
| Command words by year | Is the paper moving towards more *Explain* or more *Calculate*? |
| Paper template | How many questions, how many marks each, which topic usually comes where |
| Definitions bank | Every definition asked, with the mark-scheme wording – great for revision |
| Same wording | Parts repeated word-for-word in different sittings |

**How to read it:** there are only about 14 sittings since 2022, so a difference of one or two appearances is just noise. *Sittings %* counts each exam session once, so variants 41/42/43 of the same session are not "repeats". *Papers %* is the chance a task appears on any single paper you sit.

All tables and charts are saved in `reports/`.

In [ ]:
pool = df[df.current & df.extraction_ok].copy()
assert not pool.empty, "No checked papers from SYLLABUS_START onward - look at the extraction table in Step 5."
N_PAPERS, N_SITTINGS = pool.paper_id.nunique(), pool.sitting_idx.nunique()
print(f"Analysing {N_PAPERS} papers from {N_SITTINGS} sittings ({pool.sitting.iloc[pool.sitting_idx.argmin()]} to {pool.sitting.iloc[pool.sitting_idx.argmax()]})")
BLUE = "#2a78d6"

def tidy(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)

def question_table(data):
    """One row per question: its main topic (the topic with most marks), total marks and sitting."""
    t = data.groupby(["paper_id", "question_number", "topic"]).marks.sum().reset_index()
    main = t.sort_values("marks").groupby(["paper_id", "question_number"]).tail(1).drop(columns="marks")
    totals = data.groupby(["paper_id", "question_number"]).agg(marks=("marks", "sum"), sitting_idx=("sitting_idx", "first"))
    return main.merge(totals.reset_index(), on=["paper_id", "question_number"])

def lo_table(data):
    """How often each learning outcome is tested, and how long ago it was last tested."""
    g = data.groupby("lo")
    t = pd.DataFrame({"sittings_%": g.sitting_idx.nunique() / data.sitting_idx.nunique() * 100,
                      "papers_%": g.paper_id.nunique() / data.paper_id.nunique() * 100,
                      "marks_when_tested": g.marks.sum() / g.paper_id.nunique(),
                      "last_seen": g.sitting_idx.max(),
                      "usual_command": g.command.agg(lambda c: c.mode().iat[0])})
    t["gap"] = data.sitting_idx.max() - t.last_seen          # sittings since it was last tested
    return t.join(info[["topic", "outcome"]]).sort_values("sittings_%", ascending=False)

# 1. Important topics
paper_marks = pool.pivot_table(index=["paper_id", "year"], columns="topic", values="marks", aggfunc="sum", fill_value=0)
topics = pd.DataFrame({"avg_marks_per_paper": paper_marks.mean(),
                       "papers_with_topic_%": (paper_marks > 0).mean() * 100}).round(1).sort_values("avg_marks_per_paper")
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(topics.index, topics.avg_marks_per_paper, color=BLUE, height=0.6)
for y, v in enumerate(topics.avg_marks_per_paper):
    ax.text(v + 0.2, y, f"{v:.1f}", va="center", fontsize=8)
ax.set(title="Average marks per paper by topic", xlabel="marks (out of 100)")
tidy(ax); fig.tight_layout(); fig.savefig(REPORTS / "topic_marks.png", dpi=150); plt.show()

by_year = paper_marks.groupby(level="year").mean().T.reindex([t for t in TOPIC_ORDER if t in paper_marks.columns])
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(by_year.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(by_year.columns)), by_year.columns)
ax.set_yticks(range(len(by_year.index)), by_year.index)
for (y, x), v in np.ndenumerate(by_year.values):
    ax.text(x, y, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > by_year.values.max() * 0.6 else "black")
ax.set_title("Average marks per paper: topic by year")
fig.tight_layout(); fig.savefig(REPORTS / "topic_by_year.png", dpi=150); plt.show()

# 2. Most tested learning outcomes
los_stats = lo_table(pool)
show = los_stats.assign(last_seen=los_stats.last_seen.map(sitting_label)).round(1)
print("Most tested learning outcomes (current syllabus):")
with pd.option_context("display.max_colwidth", 80):
    display(show.head(25)[["topic", "outcome", "sittings_%", "papers_%", "marks_when_tested", "last_seen", "usual_command"]])
untested = sorted(set(info.index[~info.index.str.startswith("L")]) - set(los_stats.index))
print("Never tested since", SYLLABUS_START, ":", ", ".join(untested) or "none")

# 3. Repeated tasks = learning outcome + command word
tasks = (pool.groupby(["lo", "command"]).agg(sittings=("sitting_idx", "nunique"), papers=("paper_id", "nunique"),
                                              marks=("marks", "sum"), example=("question_id", "last"))
              .reset_index().sort_values(["sittings", "papers"], ascending=False))
tasks["task"] = tasks.command + ": " + tasks.lo.map(info.outcome).str.slice(0, 70)
print(f"\nMost repeated tasks (out of {N_SITTINGS} sittings):")
with pd.option_context("display.max_colwidth", 90):
    display(tasks.head(20)[["task", "sittings", "papers", "example"]])

# 4. Command words over time (share of marks)
cmd = pool.pivot_table(index="year", columns="command", values="marks", aggfunc="sum", fill_value=0)
cmd = cmd.div(cmd.sum(axis=1), axis=0) * 100
main_cmds = cmd.mean().nlargest(6).index
fig, ax = plt.subplots(figsize=(7, 3.5))
for c, colour in zip(main_cmds, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]):
    ax.plot(cmd.index, cmd[c], marker="o", lw=2, color=colour, label=c)
ax.set(title="Share of marks by command word", ylabel="% of marks")
ax.set_xticks(cmd.index)
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5)); tidy(ax)
fig.tight_layout(); fig.savefig(REPORTS / "command_words.png", dpi=150); plt.show()

# 5. Paper template
qs = question_table(pool)
template = qs.groupby("question_number").agg(papers=("paper_id", "nunique"), avg_marks=("marks", "mean"),
                                             usual_topic=("topic", lambda t: t.mode().iat[0]),
                                             how_often=("topic", lambda t: f"{(t == t.mode().iat[0]).mean():.0%}"))
print(f"\nA paper usually has {qs.groupby('paper_id').size().median():.0f} questions. Typical layout:")
display(template.round(1))

# 6. Definitions bank and identical wording
defs = (pool[pool.command == "Define"].sort_values("sitting_idx")
            .groupby("lo").agg(sittings=("sitting_idx", "nunique"), wording=("text", "last"), mark_scheme=("answer", "last"))
            .join(info.outcome).sort_values("sittings", ascending=False))
defs["wording"] = defs.wording.map(lambda t: squash(re.sub(r"\[(Total:)?\s*\d+\]", "", t)))
defs["mark_scheme"] = defs.mark_scheme.map(squash)
print("\nDefinitions bank (most asked first):")
with pd.option_context("display.max_colwidth", 70):
    display(defs.head(15)[["sittings", "wording", "mark_scheme"]])

key = pool.text.str.replace(r"\[(Total:)?\s*\d+\s*\]|^\s*(\(\w+\)\s*)+", "", regex=True).map(squash).str.lower()
same = (pool.assign(key=key)[key.str.len() > 25].groupby("key")
            .agg(sittings=("sitting_idx", "nunique"), ids=("question_id", list)).query("sittings > 1")
            .sort_values("sittings", ascending=False))
print(f"\n{len(same)} parts were repeated word-for-word in different sittings:")
with pd.option_context("display.max_colwidth", 80):
    display(same.head(10))

for name, table in {"topics": topics, "learning_outcomes": show, "repeated_tasks": tasks, "template": template,
                    "definitions_bank": defs, "same_wording": same}.items():
    table.to_csv(REPORTS / f"{name}.csv")

## Step 10 · Predict the next paper and test the method on past exams
**How the prediction works** (simple rules you can explain):
1. **Shape:** copy the shape of the most *typical* real paper. That is the past paper with the usual number of questions whose marks per topic are closest to the average paper. The new paper gets the same number of questions, the same topics in the same order, and the same marks per question. Each question also keeps the real **part structure**, for example `(a)[2] Define, (b)(i)[3] Calculate, (b)(ii)[2] Show that`, so the AI only has to write the content.
   - This keeps the real structure. For example, thermal physics is split over three small syllabus topics (Temperature, Ideal gases, Thermodynamics) but still fills about two questions on every paper.
   - Two questions on the same topic (often Magnetic fields) are kept too.
2. **Outcomes:** for each question, choose 2–4 learning outcomes of its main topic, with the best score:
   `score = share of sittings that tested the outcome + recency × (sittings since it was last tested, up to 3) / 3`.
   An outcome is used only once in the paper. Mixing topics inside one question gave confused questions (an a.c. supply with capacitor discharge), so each question stays on one topic.

**Backtest:** pretend each of the last 4 sittings has not happened yet. Predict it from the earlier sittings only, then check what share of the predicted outcomes really appeared on each paper.
- The table compares three recency weights against **random guessing**.
- The best weight is then used for the real prediction.
- Report these numbers in your project: they show how much the patterns really help.

In [ ]:
def typical_paper(data):
    """The past paper (with the usual number of questions) whose marks per topic are closest to the average paper."""
    sizes = data.groupby("paper_id").question_number.nunique()
    usual = sizes.index[sizes == sizes.mode().iat[0]]
    topic_marks = data.pivot_table(index="paper_id", columns="topic", values="marks", aggfunc="sum", fill_value=0)
    return ((topic_marks.loc[usual] - topic_marks.mean()) ** 2).sum(axis=1).idxmin()

def make_blueprint(data, recency):
    """Copy the shape of the typical paper, then choose the learning outcomes each question tests."""
    lt = lo_table(data)
    lt["score"] = lt["sittings_%"] / 100 + recency * lt.gap.clip(upper=3) / 3
    shape = data[data.paper_id == typical_paper(data)]
    rows, used = [], set()
    for number, q in shape.groupby("question_number"):
        topic = q.groupby("topic").marks.sum().idxmax()                     # the question's main topic
        options = lt[lt.topic == topic]
        fresh = options[~options.index.isin(used)]                          # each outcome once per paper, if possible
        marks = int(q.marks.sum())
        chosen = (fresh if len(fresh) else options).nlargest(int(np.clip(round(marks / 3.5), 2, 4)), "score")
        used |= set(chosen.index)
        skeleton = [(r.part[len(str(number)):] or "(a)", int(r.marks), r.command) for r in q.itertuples()]
        rows.append({"question": number, "topic": topic, "marks": marks,
                     "outcomes": chosen.index.tolist(), "outcome_text": chosen.outcome.tolist(),
                     "skeleton": skeleton, "parts": " ".join(f"{l}[{m}]" for l, m, _ in skeleton)})
    return pd.DataFrame(rows)

def backtest(data, weights=(0.0, 0.3, 0.6), n_last=4):
    rows = []
    for s in sorted(data.sitting_idx.unique())[-n_last:]:
        train, test = data[data.sitting_idx < s], data[data.sitting_idx == s]
        if train.sitting_idx.nunique() < 3:
            continue
        seen = set(train.lo)
        real = [set(p.lo) for _, p in test.groupby("paper_id")]   # outcomes on each real paper of that sitting
        row = {"sitting": sitting_label(s), "random guess": np.mean([len(r & seen) / len(seen) for r in real])}
        for w in weights:
            predicted = set(chain.from_iterable(make_blueprint(train, w).outcomes))
            row[f"recency {w}"] = np.mean([len(r & predicted) / len(predicted) for r in real])
        rows.append(row)
    return (pd.DataFrame(rows).set_index("sitting") * 100).round(1)

bt = backtest(pool)
print("% of predicted learning outcomes that really appeared on a paper of that sitting:")
display(pd.concat([bt, bt.mean().to_frame("average").T.round(1)]))
RECENCY = float(bt.drop(columns="random guess").mean().idxmax().split()[-1]) if len(bt) else 0.3
print(f"Using recency weight {RECENCY} for the prediction.")
bt.to_csv(REPORTS / "backtest.csv")

blueprint = make_blueprint(pool, RECENCY)
print(f"\nBlueprint for {TARGET_EXAM} ({blueprint.marks.sum()} marks), "
      f"shaped like {typical_paper(pool)} (the most typical past paper):")
with pd.option_context("display.max_colwidth", 120):
    display(blueprint[["question", "topic", "marks", "outcomes", "parts"]])
blueprint.to_csv(REPORTS / "blueprint.csv", index=False)

## Step 11 · Write the predicted paper (RAG generation with checks)
**The idea:** the AI model writes the physics, and Python does everything that must be exact.

For each blueprint question:
1. **Retrieve** up to 3 real past questions on the same topic, preferring ones that test the chosen outcomes, from different sittings, with their mark schemes. They are shown to the model for style only.
2. **Generate.** The model fills in a JSON template that already has the right parts and one line per mark, e.g. `(a)[2] Define, (b)(i)[3] Calculate`. It never works out numbers itself. Each calculation is a **Python expression**, and a graph is a **formula** `y = f(x)`, because language models often get arithmetic wrong.
3. **Finish with Python:**
   - labels and marks come from the blueprint (extra parts are dropped), so the marks are always right;
   - each expression is calculated with the Cambridge data values. A part can use an earlier answer, e.g. `ans_b_i`;
   - Python writes the mark scheme codes itself: `C1` for each working step and `A1` for the calculated answer, `B1` for each marking point of an explanation (with "max." when there are more points than marks);
   - checks: enough marking points; the value in a "Show that" part matches its calculation (75 mA and 0.075 A count as equal); no calculation or explanation part is a near-copy of a past part.
4. **Examiner review** (`REVIEW = True`): once Python finds nothing wrong, the model reads the finished question as a senior examiner and lists serious physics errors, if any.
5. **Fix, don't restart:** any problem is sent back with the question, and the model corrects only that. There are up to `MAX_TRIES` attempts in total; the best version is kept, and anything still wrong is printed as **CHECK** in the mark scheme.

**Reading the progress lines:** each attempt prints one line, such as `Q3 attempt 1: 1 problem(s) (70 s)`, followed by the problems. Problems found by the review start with `examiner:`.

**Time:** about 1–2 minutes per attempt on a T4 with the 9B model, so roughly 20–40 minutes for the paper. Good answers are cached, so re-running is instant.

In [ ]:
from pydantic import field_validator

REVIEW = True                      # a second pass: the model checks the physics of each finished question

class Part(BaseModel):
    label: str = ""                # "(a)", "(b)(i)", ... (set by Python from the blueprint)
    text: str
    mark_scheme: list[str]         # marking points or working steps; Python adds the B1 / C1 / A1 codes
    unit: str = ""                 # unit of the numerical answer, e.g. "m s^-1"
    calculation: str = ""          # ONE Python expression that gives the numerical answer
    marks: int = 0                 # set by Python from the blueprint
    answer: float | None = None    # set by Python from the calculation

    @field_validator("mark_scheme", mode="before")
    @classmethod
    def lines_from_text(cls, v):   # one long string -> one line per point
        return [line.strip() for line in re.split(r"\n|;", v) if line.strip()] if isinstance(v, str) else v

class NewQuestion(BaseModel):
    stem: str
    figure: str = ""               # words describing a diagram, if one is needed
    graph_x_label: str = ""        # a graph, if one is needed: axis labels, x range, and y as a formula of x
    graph_y_label: str = ""
    graph_x_from: float = 0
    graph_x_to: float = 0
    graph_y_formula: str = ""
    parts: list[Part]

class Review(BaseModel):
    problems: list[str] = []

DATA_SHEET = {"g": 9.81, "c": 3.00e8, "e": 1.60e-19, "u": 1.66e-27, "m_p": 1.67e-27, "m_e": 9.11e-31, "N_A": 6.02e23,
              "R": 8.31, "k": 1.38e-23, "G": 6.67e-11, "eps0": 8.85e-12, "h": 6.63e-34, "sigma": 5.67e-8}
SAFE = {**DATA_SHEET, "pi": math.pi, "abs": abs, **{f: getattr(math, f) for f in
        ["sqrt", "exp", "log", "log10", "sin", "cos", "tan", "asin", "acos", "atan", "atan2", "radians", "degrees"]}}
CALC_WORDS = {"Calculate", "Determine", "Show that", "Estimate", "Deduce"}
EVERYDAY = {0.5, 1.5, 0.25, 0.693, 24, 60, 100, 180, 273, 273.15, 360, 365, 3600}   # numbers a calculation may always use
NO_NUMBER = {"Define", "Explain", "Describe", "Suggest", "Sketch", "Draw", "Outline", "Compare", "Discuss", "Identify"}
NUMBER = re.compile(r"[-+−–]?\d*\.?\d+(?:[eE][-+]?\d+)?(?:\s*[×x]\s*10\s*\^\s*[-+−–]?\d+)?")
STATED = re.compile(r"(?:\bis|\bof|\bbe|=)\s+(?:about\s+|approximately\s+)?(" + NUMBER.pattern + ")")

def numbers_in(text):
    """All numbers in a text, e.g. 'is 12 000 K and 2.5 × 10^3 Pa' -> [12000.0, 2500.0]."""
    text = re.sub(r"(\d) (?=\d{3}\b)", r"\1", text)                 # Cambridge writes 12 000 for 12000
    return [to_number(n) for n in NUMBER.findall(text)]

def to_number(text):
    """Read a number written as text: '19.6' -> 19.6, '2.5 × 10^3' -> 2500.0."""
    text = text.replace("−", "-").replace("–", "-")
    m = re.match(r"\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)(?:\s*[×x]\s*10\s*\^\s*([-+]?\d+))?", text)
    return float(m[1]) * 10.0 ** int(m[2] or 0) if m else None

def nice(x):
    """A number written the Cambridge way, to 3 significant figures: 0.0125, 47.6, 8.94 × 10^6."""
    if x != 0 and not 0.01 <= abs(x) < 10000:
        mantissa, power = f"{x:.2e}".split("e")
        return f"{mantissa} × 10^{int(power)}"
    return f"{float(f'{x:.3g}'):g}"

def same_value(a, b):
    """True if a and b agree within 5%, also when one has a unit prefix (75 mA and 0.075 A)."""
    return any(math.isclose(a * 1000.0 ** k, b, rel_tol=0.05) for k in range(-4, 5))

def clean_expression(expression):
    """Remove common slips from a model's Python expression: 'import math; math.sqrt(2^2)' -> 'sqrt(2**2)'."""
    return re.sub(r"import \w+;?|\b(math|np|numpy)\.", "", expression).replace("^", "**").strip()

def run(expression, names={}):
    """Calculate a Python expression safely, with the data values (and earlier answers or x)."""
    return float(eval(clean_expression(expression), {"__builtins__": {}}, {**SAFE, **names}))

def pretty(expression):
    """'(120*180)/(2.5*ans_b_i)' -> '(120 × 180)/(2.5 × (b)(i))', used as the substitution step of a mark scheme."""
    expression = re.sub(r"ans_(\w+)", lambda m: "".join(f"({t})" for t in m[1].split("_")), clean_expression(expression))
    return re.sub(r"\s+", " ", expression.replace("**", "^").replace("*", " × ")).strip()

def plain(line):
    """A mark-scheme line without its mark code(s): 'v = 2 pi r / T C1' -> 'v = 2 pi r / T'."""
    code = r"[\(\[]?[ABCM]\d[\)\]]?[:.]?"
    return re.sub(rf"^\s*{code}\s*$|^(\s*{code}\s)+|(\s+{code})+\s*$", "", line).strip()

def unknown_numbers(calculation, given):
    """Numbers in a calculation that are not in the question's data (or the data sheet): a sign of a made-up value."""
    found = [float(x) for x in re.findall(r"(?<![\w.])\d+\.?\d*(?:[eE][-+]?\d+)?", calculation)]
    return [x for x in found if x and not (x == int(x) and x <= 12) and x not in EVERYDAY
            and not math.log10(x).is_integer() and not any(same_value(x, g) for g in given)]

def merge(steps, n):
    """At most n steps: extra steps are joined into the first one, so no marking point is lost."""
    if n <= 0:
        return []
    k = max(len(steps) - n + 1, 1)
    return ["; ".join(steps[:k])] + steps[k:]

def examples_for(topic, outcomes, n=3):
    """Real past questions on this topic (with mark schemes), best matching the chosen outcomes, from different sittings."""
    qs = question_table(pool)
    qs = qs[qs.topic == topic].copy()
    hits = pool[pool.lo.isin(outcomes)].groupby(["paper_id", "question_number"]).size()
    qs["hits"] = [hits.get((p, q), 0) for p, q in zip(qs.paper_id, qs.question_number)]
    qs = qs.sort_values(["hits", "sitting_idx"], ascending=False).drop_duplicates("sitting_idx").head(n)
    examples = []
    for p, q in zip(qs.paper_id, qs.question_number):
        parts = pool[(pool.paper_id == p) & (pool.question_number == q)]
        first = parts.iloc[0]
        scheme = "\n".join(f"{r.part}: {squash(r.answer, 250)}" for r in parts.itertuples())
        question = re.sub(r"\n{2,}", "\n", first.full_question)[:2000]
        examples.append(f"### Past question ({first.sitting}, paper {first.component}, Q{q})\n{question}\n\nMark scheme:\n{scheme}")
    return examples

def reply_template(row):
    """The exact JSON to fill in: the blueprint's parts, with one mark-scheme line per mark."""
    parts = []
    for label, marks, command in row.skeleton:
        numeric = command in {"Calculate", "Determine", "Estimate", "Deduce", "Show that"}
        steps = marks - 1 if numeric and command != "Show that" else marks
        word = command if command not in {"Other", "Complete", "Plot", "Draw"} else ""
        parts.append({"label": label, "text": f"{word} ...".strip(),
                      "mark_scheme": [f"working step {i + 1}" if numeric else f"marking point {i + 1}" for i in range(steps)],
                      "unit": "..." if numeric else "", "calculation": "Python expression" if numeric else ""})
    return json.dumps({"stem": "...", "figure": "", "graph_x_label": "", "graph_y_label": "", "graph_x_from": 0,
                       "graph_x_to": 0, "graph_y_formula": "", "parts": parts}, ensure_ascii=False)

def question_prompt(row, examples):
    outcomes = "\n".join(f"- {t}" for t in row.outcome_text)
    constants = ", ".join(f"{k} = {v:.3g}" for k, v in DATA_SHEET.items())
    return f"""You are an experienced Cambridge International A Level Physics (9702) examiner.
Write ONE new question for a Paper 4 (A Level Structured Questions) practice paper.

Topic: {row.topic}
Learning outcomes to assess:
{outcomes}

Rules:
1. Use a NEW situation and NEW numbers. The past questions below are only for style: do not copy their sentences.
2. The stem gives the situation and all the data the parts need. It never contains the part questions.
3. Cambridge data values you may use: {constants}.
4. Write powers as 10^5 and units as m s^-1. Do not use LaTeX.
5. Mark scheme: one line per marking point or working step, WITHOUT mark codes (Python adds B1, C1 and A1).
   A part with 2 marks needs 2 separate marking points.
6. Never work out numbers yourself. For a numerical answer write ONE Python expression in "calculation" (numbers, the
   data names above, sqrt, exp, log, sin, cos, pi, powers with **) and the unit in "unit". Its mark scheme lists only the
   working steps; Python adds the final answer line. To use the answer of an earlier part write ans_ and its label, e.g. ans_b_i.
7. In a "Show that" part, state the value in the text (e.g. "Show that the speed is about 4.2 m s^-1") and give the
   calculation that gives it.
8. Never refer to a table, graph or diagram that you do not give. Describe a needed diagram in "figure". Give a graph only
   if a part needs values read from it: graph_y_formula is a Python expression of x, drawn from graph_x_from to graph_x_to.
   Never draw the answer to a Sketch part.
9. Every part must be correct physics and answerable from the data given. Keep each part under 60 words.

Past questions on this topic (real exams with their mark schemes, for style only):

{chr(10).join(examples)}

Your question must have exactly these parts and marks ({row.marks} marks in total). Fill in this JSON:
{reply_template(row)}
"""

def fix_prompt(qn, problems, row):
    return ("You wrote this Cambridge International A Level Physics (9702) exam question. Python has already added the "
            "mark codes and the calculated answers:\n"
            + qn.model_dump_json(exclude={"parts": {"__all__": {"marks", "answer"}}}) +
            "\n\nIt has these problems:\n" + "\n".join(f"- {p}" for p in problems) +
            f"\n\nThe parts must be exactly: {', '.join(f'{l} [{m}]' for l, m, _ in row.skeleton)}. "
            "Return the whole question as corrected JSON in the same format. Change only what is needed to fix the problems.")

def finish_question(qn, row):
    """Python does the exact work: labels and marks from the blueprint, answers from the calculations,
    and the mark codes. Returns the problems that the model still has to fix."""
    qn.stem = re.split(r"\s\((?:[a-h]|i{1,3}|iv|v)\)\s", qn.stem)[0].strip()       # the stem must not contain parts
    labels = [label for label, _, _ in row.skeleton]
    given = {p.label.replace(" ", ""): p for p in qn.parts}
    if all(label in given for label in labels):
        qn.parts = [given[label] for label in labels]                               # extra parts are dropped
    elif len(qn.parts) != len(labels):
        return [f"the question must have exactly these {len(labels)} parts: "
                + ", ".join(f"{l} [{m}]" for l, m, _ in row.skeleton)]
    problems, answers = [], {}
    text = " ".join([qn.stem, qn.figure] + [p.text for p in qn.parts])
    given = [abs(x) for x in numbers_in(text) if x] + list(DATA_SHEET.values())
    if qn.graph_y_formula:
        try:
            if qn.graph_x_to <= qn.graph_x_from:
                raise ValueError("graph_x_to must be larger than graph_x_from")
            run(qn.graph_y_formula, {"x": qn.graph_x_from}), run(qn.graph_y_formula, {"x": qn.graph_x_to})
        except Exception as e:
            problems.append(f"the graph does not work ({e}); give graph_y_formula as one Python expression of x")
            qn.graph_y_formula = ""
    for p, (label, marks, _) in zip(qn.parts, row.skeleton):
        p.label, p.marks, p.answer = label, marks, None
        command = command_word(p.text)
        if p.calculation and command not in NO_NUMBER:
            try:
                p.answer = run(p.calculation, answers)
                answers["ans_" + "_".join(re.findall(r"\w+", label))] = p.answer
                for x in unknown_numbers(p.calculation, given):
                    problems.append(f"part {label}: the calculation uses {nice(x)}, which is not in the question's data; "
                                    "give it in the stem, or write ans_ and the label to use an earlier answer")
            except Exception as e:
                if command in CALC_WORDS:                               # elsewhere a broken calculation is just dropped
                    problems.append(f"part {label}: the calculation does not run ({e}). Use one expression with numbers, "
                                    "the data names, sqrt, exp, log, sin, cos, pi and ans_ names of earlier parts")
        steps = [plain(line) for line in p.mark_scheme]
        if p.answer is not None and command != "Show that":           # a calculation: C1 steps, then Python's answer A1
            finals = [i for i, line in enumerate(p.mark_scheme) if re.search(r"A1\W*$", line)]
            steps = [s for s in (steps[:finals[0]] if finals else steps) if s]
            steps = [s for s in steps if "=" in s or not any(same_value(to_number(n), p.answer) for n in NUMBER.findall(s))]
            if len(steps) < marks - 1 and "= " + pretty(p.calculation) not in steps:
                steps.append("= " + pretty(p.calculation))              # the substitution step
            steps = merge(steps, marks - 1)
            p.mark_scheme = [f"{s} C1" for s in steps] + [f"{nice(p.answer)} {p.unit} A1".replace("  ", " ")]
            if len(steps) < marks - 1:
                problems.append(f"part {label} has {marks} marks: give {marks - 1} working steps before the answer")
            continue
        steps = list(dict.fromkeys(s for s in steps if s))                 # empty and repeated points do not count
        if command == "Show that":                                      # working steps, the last one A1
            stated = [to_number(n) for n in STATED.findall(re.sub(r"(\d) (?=\d{3}\b)", r"\1", p.text))]
            if p.answer is not None and stated and not any(same_value(s, p.answer) for s in stated):
                problems.append(f"part {label}: the value to show in the question does not match its calculation, "
                                f"which gives {nice(p.answer)} {p.unit}; change the value in the text or the data")
            substitution = f"= {pretty(p.calculation)} = {nice(p.answer)} {p.unit}".strip() if p.answer is not None else ""
            if len(steps) < marks and substitution and substitution not in steps:
                steps.append(substitution)
            steps = merge(steps, marks)
            p.mark_scheme = [f"{s} C1" for s in steps[:-1]] + [f"{s} A1" for s in steps[-1:]]
        else:                                                           # marking points, B1 each
            p.mark_scheme = [f"{s} B1" for s in steps] + ([f"max. {marks}"] if len(steps) > marks else [])
        if len(steps) < marks:
            problems.append(f"part {label} has {marks} marks but only {len(steps)} marking point(s); "
                            f"write {marks} separate points")
    return problems

def copy_check(qn):
    """Calculation and explanation parts must not be near-copies of past parts (definitions may repeat)."""
    texts = [part_text(qn.stem, p.text, " ".join(p.mark_scheme)) for p in qn.parts]
    closest = (embed(texts) @ part_vecs.T).max(axis=1)
    problems = [f"part {p.label} is almost a copy of a past question (similarity {c:.2f}); change its situation"
                for p, c in zip(qn.parts, closest) if c > COPY_LIMIT and command_word(p.text) not in {"Define", "State"}]
    return problems, float(closest.max())

def as_text(qn):
    """The question and its mark scheme as plain text, for the examiner review."""
    lines = [qn.stem] + ([f"[Figure: {qn.figure}]"] if qn.figure else [])
    if qn.graph_y_formula:
        lines.append(f"[Graph of {qn.graph_y_label} against {qn.graph_x_label}: "
                     f"y = {qn.graph_y_formula} for x from {qn.graph_x_from} to {qn.graph_x_to}]")
    for p in qn.parts:
        lines += [f"{p.label} {p.text} [{p.marks}]", "    mark scheme: " + " | ".join(p.mark_scheme)]
    return "\n".join(lines)

def review_prompt(qn):
    return ("You are a senior Cambridge International A Level Physics (9702) examiner checking a new question before "
            "it is printed:\n\n" + as_text(qn) +
            "\n\nList only SERIOUS errors, at most 3: wrong physics, a part that cannot be answered from the data given, "
            "a reference to a table or diagram that is not given, or a mark scheme that does not answer its part. "
            "The numbers in the mark scheme were calculated by Python from the question's own expressions. Ignore style. "
            'If there are no serious errors, reply {"problems": []}.\nReply format: {"problems": ["...", "..."]}')

generated = []
for row in tqdm(list(blueprint.itertuples()), desc="Writing questions"):
    prompt, best = question_prompt(row, examples_for(row.topic, row.outcomes)), None
    for attempt in range(MAX_TRIES):
        start = time.time()
        try:
            qn = ask_llm(prompt, NewQuestion, temperature=0.6 if best is None else 0.4, attempt=attempt, max_new_tokens=3000)
        except ValueError as e:
            print(f"Q{row.question} attempt {attempt + 1}: unusable reply ({time.time() - start:.0f} s) - {e}")
            continue
        problems = finish_question(qn, row)
        copies, closest = copy_check(qn)
        problems += copies
        if not problems and REVIEW:                    # Python found nothing wrong: now an examiner reads it
            try:
                problems = ["examiner: " + p for p in ask_llm(review_prompt(qn), Review, max_new_tokens=400).problems[:3]]
            except ValueError as e:
                print("  (the review reply could not be read:", e, ")")
        print(f"Q{row.question} attempt {attempt + 1}: {len(problems)} problem(s) ({time.time() - start:.0f} s)"
              + "".join(f"\n    - {p}" for p in problems))
        if best is None or len(problems) <= len(best["problems"]):     # ties: the newer version is kept
            best = {"row": row, "question": qn, "problems": problems, "closest_past": closest}
        if not problems:
            break
        prompt = fix_prompt(best["question"], best["problems"], row)    # the next attempt corrects the best version
    if best is None or not all(p.marks for p in best["question"].parts):
        print(f"Q{row.question} ({row.topic}) could not be generated - re-run this cell to try again.")
    else:
        generated.append(best)

display(pd.DataFrame([{"question": g["row"].question, "topic": g["row"].topic, "marks": sum(p.marks for p in g["question"].parts),
                       "problems left": len(g["problems"]), "closest past part": round(g["closest_past"], 2)} for g in generated]))

## Step 12 · Show and save the paper and mark scheme
**What it does:** draws each graph from its formula with matplotlib, shows the paper here, and saves four files in `reports/`:
- `predicted_paper.docx` and `predicted_mark_scheme.docx`, ready to print;
- `.md` copies of both.

Any problem the checks could not fix is printed in the mark scheme as **CHECK**. Solve the paper yourself before trusting it: a human check is the final step.

In [ ]:
from docx import Document
from docx.shared import Inches

def graph_png(q, qn):
    """Draw the question's graph from its formula y = f(x), so the curve is smooth and exact."""
    path = REPORTS / f"figure_{q}.png"
    xs = np.linspace(qn.graph_x_from, qn.graph_x_to, 300)
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(xs, [run(qn.graph_y_formula, {"x": x}) for x in xs], color=BLUE, lw=2)
    ax.set(title=f"Figure {q}.2", xlabel=qn.graph_x_label, ylabel=qn.graph_y_label)
    ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)
    return path

title = f"Predicted Paper 4 - {TARGET_EXAM}"
note = "Practice paper written by an AI model from past-paper patterns. It is not an official Cambridge paper."
qp, ms = Document(), Document()
qp.add_heading(title, 0); qp.add_paragraph(note)
ms.add_heading(f"Mark scheme - {title}", 0)
qp_md, ms_md = [f"# {title}\n*{note}*\n"], [f"# Mark scheme - {title}\n"]

for q, g in enumerate(generated, 1):                 # numbered 1, 2, 3 ... even if a question was skipped
    qn = g["question"]
    para = qp.add_paragraph()
    para.add_run(f"{q}   ").bold = True
    para.add_run(qn.stem)
    qp_md.append(f"\n**{q}** {qn.stem}\n")
    if qn.figure:
        qp.add_paragraph().add_run(f"[Figure {q}.1: {qn.figure}]").italic = True
        qp_md.append(f"*[Figure {q}.1: {qn.figure}]*\n")
    if qn.graph_y_formula:
        path = graph_png(q, qn)
        qp.add_picture(str(path), width=Inches(4.5))
        qp_md.append(f"![Figure {q}.2]({path.name})\n")
    for p in qn.parts:
        qp.add_paragraph(f"{p.label} {p.text}    [{p.marks}]")
        qp_md.append(f"{p.label} {p.text} **[{p.marks}]**\n")
    total = sum(p.marks for p in qn.parts)
    qp.add_paragraph(f"[Total: {total}]")
    qp_md.append(f"*[Total: {total}]*\n")

    table = ms.add_table(rows=1, cols=3)
    table.style = "Table Grid"
    for cell, head in zip(table.rows[0].cells, ["Question", "Answer", "Marks"]):
        cell.text = head
    ms_md.append(f"\n| Question {q} | Answer | Marks |\n|---|---|---|")
    for p in qn.parts:
        cells = table.add_row().cells
        cells[0].text, cells[1].text, cells[2].text = f"{q}{p.label}", "\n".join(p.mark_scheme), str(p.marks)
        ms_md.append(f"| {q}{p.label} | {'<br>'.join(p.mark_scheme)} | {p.marks} |")
    if g["problems"]:
        ms.add_paragraph("CHECK: " + "; ".join(g["problems"]))
        ms_md.append("\n**CHECK:** " + "; ".join(g["problems"]) + "\n")

qp.save(REPORTS / "predicted_paper.docx"); ms.save(REPORTS / "predicted_mark_scheme.docx")
(REPORTS / "predicted_paper.md").write_text("\n".join(qp_md), encoding="utf-8")
(REPORTS / "predicted_mark_scheme.md").write_text("\n".join(ms_md), encoding="utf-8")

display(Markdown(f"# {title}\n*{note}*"))
for q, g in enumerate(generated, 1):                  # show the paper here, with its graphs
    qn = g["question"]
    display(Markdown(f"**{q}** {qn.stem}" + (f"\n\n*[Figure {q}.1: {qn.figure}]*" if qn.figure else "")))
    if qn.graph_y_formula:
        display(Image(str(REPORTS / f"figure_{q}.png"), width=420))
    display(Markdown("\n\n".join(f"{p.label} {p.text} **[{p.marks}]**" for p in qn.parts)))
paper_total = sum(p.marks for g in generated for p in g["question"].parts)
print(f"Total marks: {paper_total}" + ("" if paper_total == 100 else " - not 100, re-run Step 11 to retry the missing questions"))
print("Saved in", REPORTS)

## Step 13 · Ask the archive (RAG question answering)
**What it does:** `ask_archive("...")` finds the most relevant past parts with `search()`, then lets the AI model answer *only* from them, listing which question IDs it used. Use `show_question(id)` to check the originals. `similar(id)` lists the closest parts from other sittings, with no AI involved.

Change the questions in the last lines and re-run this cell as often as you like.

In [ ]:
class ArchiveAnswer(BaseModel):
    answer: str
    sources: list[str]

def ask_archive(question, years=(SYLLABUS_START, 2100), k=6):
    hits = search(question, years, k)
    evidence = "\n\n".join(f"[{r.question_id}] ({r.sitting}, {r.marks} marks)\n{r.search_text}" for r in hits.itertuples())
    prompt = (f"Answer the question using ONLY the past exam parts below. Explain the method and how the marks are "
              f"awarded. In sources, list the IDs of the parts you used.\n\nQuestion: {question}\n\nPast exam parts:\n{evidence}"
              '\n\nReply format: {"answer": "...", "sources": ["9702_s24_41_Q1(a)", "..."]}')
    try:
        out = ask_llm(prompt, ArchiveAnswer)
    except ValueError as e:
        print("The model's reply could not be used:", e)
        return
    display(Markdown(out.answer))
    used = {s.strip("[] ") for s in out.sources}
    with pd.option_context("display.max_colwidth", 80):
        display(hits.assign(used=hits.question_id.isin(used))[["question_id", "sitting", "marks", "lo", "used", "text"]])

ask_archive("How do mark schemes award marks for questions on Wien's displacement law?")
display(similar(pool.question_id.iat[0]))

## Notes, limits and ideas
- **Everything is a draft.** Syllabus tags, the prediction and the generated questions all come from models and simple rules. The backtest and your own checks of the tags show how far to trust them.
- **Figures:** the generated paper describes diagrams in words and draws graphs from data. Circuit diagrams could be drawn with the `schemdraw` library as a next step.
- **Better questions:** `Qwen/Qwen3.5-9B` is the strongest Qwen3.5 model that fits a free T4. Use `Qwen/Qwen3.5-4B` if the runtime is too slow (about twice as fast).
- **If generation fails:** read the reason on the progress line and the full reply in `llm_bad_replies.txt`. A reply that is cut off needs a larger `max_new_tokens` in Step 11; a reply in the wrong shape usually works on the next attempt.
- **Speed:** Qwen3.5 mixes normal attention with a newer "linear attention". On a T4 its fast kernels are not available, so it runs on plain PyTorch, which is slower but correct. Most of the time goes into writing the reply, so shorter replies are faster.
- **Science-fair ideas:**
  - Hand-check 40 random tags and report the accuracy.
  - Compare the backtest against random guessing.
  - Ask a teacher to rate a mix of real and generated questions without knowing which is which.